In [ ]:
import bs4

from trouver.helper import is_not_space_and_not_punc
from trouver.helper.html import HTMLTagWithIndices
from trouver.helper.latex.core import _is_balanced_braces, _first_curly_bracket, _last_curly_bracket

from trouver.helper.regex import latex_indices

In [ ]:
from trouver.machine_learning.tokenize.def_and_notat_token_classification import _ranges_overlap, _make_tag, _str_contains_latex_command_to_avoid

In [ ]:
from fastcore.test import *

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _extend_tag_data_range_for_math_mode(
        tag_tuple: HTMLTagWithIndices,
        main_text: str,
        latex_inds: list[tuple[int, int]],
        ) -> HTMLTagWithIndices:
    """
    Extend tag data so that the tag data does not start or end within latex math mode string.
    """
    extended_range = [tag_tuple.start, tag_tuple.end]
    for tex_range in latex_inds:
        if not _ranges_overlap(HTMLTagWithIndices(0, tex_range[0], tex_range[1]), tag_tuple):
            continue
        extended_range = (min(extended_range[0], tex_range[0]), max(extended_range[1], tex_range[1]))
    return _update_tag_data(tag_tuple, main_text, extended_range[0], extended_range[1])

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _extend_tag_data_range_to_border_space_or_punc(
        tag_tuple: HTMLTagWithIndices,
        main_text: str,
        ) -> HTMLTagWithIndices:
    """
    Extend tag data so that the tag data borders spaces or punctuations.

    Helper function to `_consolidate_token_preds`.
    """
    combined_range = [tag_tuple.start, tag_tuple.end]
    while combined_range[0] != 0 and not main_text[combined_range[0]-1].isspace():
        combined_range[0] -= 1
    # while combined_range[1] != len(main_text) and not main_text[combined_range[1]].isspace():
    while combined_range[1] != len(main_text) and is_not_space_and_not_punc(main_text[combined_range[1]]):
        combined_range[1] += 1
    return _update_tag_data(tag_tuple, main_text, combined_range[0], combined_range[1])



In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _extend_tag_data_ranges_to_balance_curly_braces(
        tag_tuple: HTMLTagWithIndices,
        main_text: str
        ) -> list[HTMLTagWithIndices]:
    """
    Extend tag data to balance curly braces
    """
    combined_range = [tag_tuple.start, tag_tuple.end]
    while not _is_balanced_braces(main_text[combined_range[0]:combined_range[1]]):
        changed = False
        while combined_range[0] != 0 and _first_curly_bracket(main_text[combined_range[0]:combined_range[1]]) == r'}':
            combined_range[0] -= 1
            changed = True
        while combined_range[1] != len(main_text) and _last_curly_bracket(main_text[combined_range[0]:combined_range[1]]) == r'{':
            combined_range[1] += 1
            changed = True
        if not changed:
            break
    return _update_tag_data(tag_tuple, main_text, combined_range[0], combined_range[1])



In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _extend_tag_data_ranges(
        main_text: str,
        latex_inds: list[tuple[int, int]],
        tag_data: list[HTMLTagWithIndices],
        ) -> list[HTMLTagWithIndices]:
    """
    Extend tag data so that
    1. the tag data does not start or end within any latex math mode string.
    2. the tag data is immediately preceded by whitespace (or the start/end of line)
       and followed by whitespace or punctuation
    3. the tag data does not start/end in the middle of the arguments of a latex command.

    Helper function to `_consolidate_token_preds`.
    """
    # TODO: make sure that tag has balanced curly braces 
    extended_tag_data = []
    for tag_tuple in tag_data:
        tag_tuple_before_extension = tag_tuple
        while True:
            tag_tuple_after_extension = _extend_tag_data_range_for_math_mode(
                tag_tuple_before_extension, main_text, latex_inds)
            tag_tuple_after_extension = _extend_tag_data_range_to_border_space_or_punc(
                tag_tuple_after_extension, main_text)
            tag_tuple_after_extension = _extend_tag_data_ranges_to_balance_curly_braces(
                tag_tuple_after_extension, main_text)
            tag_tuple_before_extension = tag_tuple_after_extension 
            if (tag_tuple_before_extension[1] == tag_tuple_after_extension[1]
                    and tag_tuple_before_extension[2] == tag_tuple_after_extension[2]):
                break
        extended_tag_data.append(tag_tuple_after_extension)
    return extended_tag_data


In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _update_tag_data(
        tag_tuple: HTMLTagWithIndices,
        main_text: str,
        new_start: int,
        new_end: int) -> HTMLTagWithIndices:
    new_text = main_text[new_start:new_end]
    tag_type = 'definition' if 'definition' in tag_tuple.tag.attrs else 'notation'
    new_tag = _make_tag(new_text, tag_type)
    return HTMLTagWithIndices(new_tag, new_start, new_end)


    

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
import string
def _trim_punctuation_from_indices(main_text, start, end):
    # Standard prose punctuation only.
    # Specifically EXCLUDING characters like \, {, }, $, _, ^
    prose_punc = ",.!?;: "
    # prose_punc = ",.!?;:\"'" 
    
    while end > start and main_text[end-1] in prose_punc:
        end -= 1
    while start < end and main_text[start] in prose_punc:
        start += 1
    return start, end

In [ ]:
# Test Case 1: The "Greedy Comma" (Your specific issue)
main_text = "take an analytic arc, a 1-dimensional"
# Model predicts "arc," (indices 17 to 21)
start, end = 17, 21 
test_eq(main_text[start:end], "arc,") 

new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)
test_eq(main_text[new_start:new_end], "arc")
test_eq(new_start, 17)
test_eq(new_end, 20)

# Test Case 2: The "Greedy Period" at the end of a sentence
main_text = "This is a homotopy. Next sentence..."
start, end = 10, 19 # "homotopy."
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)
test_eq(main_text[new_start:new_end], "homotopy")

# Test Case 3: Leading Punctuation (e.g. model starts too early)
main_text = "Values (x,y) are..."
start, end = 7, 11 # "(x,y"
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)

# Assuming '(' is NOT in your prose_punc, but ',' is.
# If prose_punc = ",.!?;:", then only the comma would be trimmed if it were at the edge.
# Let's test a leading comma:

# Test Case 3: Leading Punctuation AND Space
main_text = "word, word"
start, end = 4, 10 # ", word"
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)
# Since space IS in prose_punc, it gets trimmed along with the comma!
test_eq(main_text[new_start:new_end], "word") 


# Test Case 4: LaTeX Protection (The "Do No Harm" test)
main_text = r"The $\zeta(s)$ function"  # Length: 23
# Indices:
# 4  5  6  7  8  9  10 11 12 13
# $  \  z  e  t  a  (  s  )  $
start, end = 4, 14  # <--- Changed from 13 to 14
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)

test_eq(main_text[new_start:new_end], r"$\zeta(s)$")

# Test Case 5: Multiple Punctuation marks (e.g. "arc...")
main_text = "An analytic arc..."
start, end = 12, 18 # "arc..."
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)
test_eq(main_text[new_start:new_end], "arc")

# Test Case 6: The "Null" case (Tag is only punctuation)
main_text = "Hello , world"
start, end = 6, 7 # ","
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)
test_eq(new_start >= new_end, True) # Logic should discard this in consolidate_token_preds

main_text = "The category C, and its objects"
start, end = 14, 15 # ","
new_start, new_end = _trim_punctuation_from_indices(main_text, start, end)
test_eq(new_start >= new_end, True)


In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _cutoff_notation_tag_data(
        main_text: str,
        tag_data: list[HTMLTagWithIndices],
        ) -> list[HTMLTagWithIndices]:
    """
    Helper function to `_consolidate_token_preds`.

    Guarantees that a notation tag is a pure math mode latex string
    by cutting only the pure math mode string
    that occurs within it. Assumes that `_extend_tag_data_ranges_to_encompass_latex`
    works as intended.
    """
    cutout_notation_tag_data: list[HTMLTagWithIndices] = []
    for tag, start, end in tag_data:
        if not 'notation' in tag.attrs:
            cutout_notation_tag_data.append(HTMLTagWithIndices(tag, start, end))
            continue
        tag_text = main_text[start:end]
        tex_inds_in_tagged = latex_indices(tag_text)
        for sub_start, sub_end in tex_inds_in_tagged:
            tex_str = main_text[start+sub_start:start+sub_end]
            cutout_tag = _make_tag(tex_str, 'notation')
            cutout_notation_tag_data.append(
                HTMLTagWithIndices(cutout_tag, start+sub_start, start+sub_end))
    return cutout_notation_tag_data




In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _no_overlap_with_previous_tag_data(
        ultimate_tag_data: list[HTMLTagWithIndices],
        current_tag_data: HTMLTagWithIndices  # Current tag data
        ) -> bool:
    """
    Return `True`, if the `current_tag_data` does not overlap with
    any tag data that will be ultimately added. 

    Helper function for `_consolidate_token_preds`.
    """
    for prev in reversed(ultimate_tag_data):
        if _ranges_overlap(prev, current_tag_data):
            return False
    return True

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _collate_html_tags(
        tag_data_1: list[HTMLTagWithIndices],
        tag_data_2: list[HTMLTagWithIndices],
    ) -> list[tuple[bs4.element.Tag], int, int]:
    """
    Collates the lists of HTML tags and the indices within a certain text
    (which is not-needed for this function and hence not included)
    that the HTML tags need to replace.

    If there are entries in `tag_data_1` and `tag_data_2` with overlapping
    ranges, then the entry from `tag_data_1` is prioritized and the entry
    from `tag_data_2` is discarded.

    Helper function to `auto_mark_def_and_notats`
    """
    collated_list: list[HTMLTagWithIndices] = []
    i, j = 0, 0
    while i < len(tag_data_1) and j < len(tag_data_2):
        current_1 = tag_data_1[i]
        current_2 = tag_data_2[j]
        if _ranges_overlap(current_1, current_2): # Ignore current_2
            j += 1
            continue
        if current_1[1] > current_2[1]:
            collated_list.append(current_2)
            j += 1
        else:
            collated_list.append(current_1)
            i += 1
    while i < len(tag_data_1):
        collated_list.append(tag_data_1[i])
        i += 1
    while j < len(tag_data_2):
        collated_list.append(tag_data_2[j])
        j += 1
    return collated_list



In [ ]:
#| hide


tag_data_1 = [
    HTMLTagWithIndices('', 0, 1),
    HTMLTagWithIndices('', 9, 12),
    HTMLTagWithIndices('', 20, 21)
]

tag_data_2 = [
    HTMLTagWithIndices('', 2, 4),
    HTMLTagWithIndices('', 6, 7),
    HTMLTagWithIndices('', 8, 10), # This should be discarded
    HTMLTagWithIndices('', 10, 13), # This should be discarded
    HTMLTagWithIndices('', 17, 20),
    HTMLTagWithIndices('', 21, 24)
]
output = _collate_html_tags(tag_data_1, tag_data_2)
test_eq(output, [
    HTMLTagWithIndices('', 0, 1), HTMLTagWithIndices('', 2, 4), HTMLTagWithIndices('', 6, 7),
    HTMLTagWithIndices('', 9, 12), HTMLTagWithIndices('', 17, 20), HTMLTagWithIndices('', 20, 21), 
    HTMLTagWithIndices('', 21, 24)])

In [ ]:
#| hide
# TODO: test _html_tags_from_token_preds

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _consolidate_token_preds(
        main_text: str,
        tag_data: list[HTMLTagWithIndices]
        ) -> list[HTMLTagWithIndices]:
    
    # --- STEP 1: PRE-TRIM ---
    # We must discard the tag if it's JUST a comma BEFORE extending.
    clean_input_tags = []
    for td in tag_data:
        ns, ne = _trim_punctuation_from_indices(main_text, td.start, td.end)
        if ns < ne: # Only keep if something other than punc remains
            clean_input_tags.append(_update_tag_data(td, main_text, ns, ne))
    
    # --- STEP 2: EXTEND ---
    latex_inds = latex_indices(main_text)
    # Use the cleaned tags here so the comma can't "pull in" the rest of the word
    extended_tag_data = _extend_tag_data_ranges(main_text, latex_inds, clean_input_tags)
    tag_data_notats_chopped = _cutoff_notation_tag_data(main_text, extended_tag_data)
    
    # --- STEP 3: FINAL POST-TRIM ---
    ultimate_tag_data: list[HTMLTagWithIndices] = []
    for tag_point in tag_data_notats_chopped:
        new_start, new_end = _trim_punctuation_from_indices(main_text, tag_point.start, tag_point.end)
        
        if new_start >= new_end:
            continue
            
        tag_point = _update_tag_data(tag_point, main_text, new_start, new_end)
        
        if (_no_overlap_with_previous_tag_data(ultimate_tag_data, tag_point)
                and not _str_contains_latex_command_to_avoid(tag_point.tag.text)):
            ultimate_tag_data.append(tag_point)
            
    return ultimate_tag_data

In [ ]:
#| hide
main_text = 'Hi. This is some text. Here is a notation $$M_k :=... $$ and here is some more $$G_k :=...$$'

soup = bs4.BeautifulSoup('', 'html.parser')
# In actuality, the tag data will have more information, but the following
# is good enough
# for the purposes of this test
tag_1 = soup.new_tag('span', notation='')
tag_2 = soup.new_tag('span', notation='')
# The following tries to test an erroneous short/incomplete marking of the very first 
# dollar sign `$` as well as the subtext 'G_K ' of the second math mode text.

tag_data = [HTMLTagWithIndices(tag_1, 42, 43), HTMLTagWithIndices(tag_2, 81, 85)]
output = _consolidate_token_preds(main_text, tag_data)
test_eq(main_text[output[0][1]: output[0][2]], '$$M_k :=... $$')
test_eq(main_text[output[1][1]: output[1][2]], '$$G_k :=...$$')


# In the following example, the notation is a priori
# found to be 'zeta(s)$ as $$\zeta'. 
# So first, the tagged data is extended to encompass
# '$\zeta(s)$ as $$\zeta(s) = ...$$' and then pure latex
# math mode str are extracted
main_text = r'Define $\zeta(s)$ as $$\zeta(s) = ...$$  '

soup = bs4.BeautifulSoup('', 'html.parser')
tag_1 = soup.new_tag('span', notation='')
tag_data = [HTMLTagWithIndices(tag_1,9,27)]
output = _consolidate_token_preds(main_text, tag_data)
# main_text.find('zeta')
test_eq(main_text[output[0][1]:output[0][2]], r'$\zeta(s)$')
test_eq(main_text[output[1][1]:output[1][2]], r'$$\zeta(s) = ...$$')



# In the following example, we have erroneous
# notation and definition markings which start/end within the same 
# math mode string.
# In this case, the preceding marking takes precedence and 
# the latter overlapping marking is discarded;
# this is an unfortunate feature that must be implemented to get
# around shortcomings of the model.
main_text = r'The Riemann zeta function $\zeta(s)$ is defined as...'
soup = bs4.BeautifulSoup('', 'html.parser')
tag_1 = soup.new_tag('b', definition='')
tag_2 = soup.new_tag('span', notation='')
tag_data = [HTMLTagWithIndices(tag_1,4,27), HTMLTagWithIndices(tag_2,26,30)] # the tag predictions overlap at $ \zeta(s)$.
output = _consolidate_token_preds(main_text, tag_data)
test_eq(main_text[output[0][1]:output[0][2]], r'Riemann zeta function $\zeta(s)$')

# In the following example, the definition tag prediction ends within the argument
# of the \emph command. 
main_text = r'The \emph{arc complex}'
soup = bs4.BeautifulSoup('', 'html.parser')
tag = soup.new_tag('b', definition="")
tag_data = [HTMLTagWithIndices(tag,4,21)]
output = _consolidate_token_preds(main_text, tag_data)
test_eq(main_text[output[0][1]:output[0][2]], r'\emph{arc complex}')

# In the following example, the definition tag prediction ends amidst of the \emph command. 
main_text = r'The \emph{arc complex}'
soup = bs4.BeautifulSoup('', 'html.parser')
tag = soup.new_tag('b', definition="")
tag_data = [HTMLTagWithIndices(tag,6,8)]
output = _consolidate_token_preds(main_text, tag_data)
test_eq(main_text[output[0][1]:output[0][2]], r'\emph{arc complex}')

# In the following example, a bad command gets predicted. 
# The consolidation should thus leave out this prediction.
main_text = r'\section{lalalala} We define a homotopy'
soup = bs4.BeautifulSoup('', 'html.parser')
tag = soup.new_tag('b', definition="")
tag_data = [HTMLTagWithIndices(tag,0,2)]
output = _consolidate_token_preds(main_text, tag_data)
test_eq(len(output), 0)
# print(output)

# In the following example, the definition tag prediction ends with a punctuation mark
# (a period in this case). 
main_text = r'This is called a homotopy. There exist...'
soup = bs4.BeautifulSoup('', 'html.parser')
tag = soup.new_tag('b', definition="")
tag_data = [HTMLTagWithIndices(tag,17,24)]
output = _consolidate_token_preds(main_text, tag_data)
test_eq(main_text[output[0][1]:output[0][2]], r'homotopy')

# The `_extend_tag_data_range_to_border_space_or_punc` function should not try to
# encapsulate the period as part of the definition tag.
tag_data = [HTMLTagWithIndices(tag,18,23)]
output = _consolidate_token_preds(main_text, tag_data)
test_eq(main_text[output[0][1]:output[0][2]], r'homotopy')











# --- Test Case: Greedy Comma in Definition ---
# Scenario: Model predicts "analytic arc," including the trailing comma.
main_text = "Take an analytic arc, a 1-dimensional manifold."
soup = bs4.BeautifulSoup('', 'html.parser')
tag = soup.new_tag('b', definition="")
# Indices 8 to 21 is "analytic arc,"
tag_data = [HTMLTagWithIndices(tag, 8, 21)] 
output = _consolidate_token_preds(main_text, tag_data)

# Should trim the comma but keep the words
test_eq(main_text[output[0].start: output[0].end], "analytic arc")


# --- Test Case: Multiple Greedy Punctuation Marks ---
# Scenario: Model captures trailing ellipsis or multiple dots.
main_text = "We define the concept of a topos..."
tag = soup.new_tag('b', definition="")
# Indices 27 to 35 is "topos..."
tag_data = [HTMLTagWithIndices(tag, 27, 35)]
output = _consolidate_token_preds(main_text, tag_data)

test_eq(main_text[output[0].start: output[0].end], "topos")


# --- Test Case: Mixed Punctuation and LaTeX ---
# Scenario: A notation followed by prose punctuation where the model over-captures.
main_text = r"Let the set be $X$."
tag = soup.new_tag('span', notation="")
# Indices 15 to 19 is "$X$."
tag_data = [HTMLTagWithIndices(tag, 15, 19)]
output = _consolidate_token_preds(main_text, tag_data)

# Trimming should remove the period but keep the math mode delimiter
test_eq(main_text[output[0].start: output[0].end], r"$X$")


# --- Test Case: Only Punctuation Predicted ---
# Scenario: Model hallucinations that result in just a comma being marked.
main_text = "The category C, and its objects"
tag = soup.new_tag('b', definition="")
# Index 14 to 15 is just the comma ","
tag_data = [HTMLTagWithIndices(tag, 14, 15)]
output = _consolidate_token_preds(main_text, tag_data)

# The trim function should empty this, and consolidate should discard it.
test_eq(len(output), 0)


# --- Test Case: Parentheses at Edges ---
# Scenario: Many mathematical definitions are written as (Concept). 
# If parentheses are in your prose_punc, they should be trimmed.
main_text = "We call this a (semi-group)."
tag = soup.new_tag('b', definition="")
# Indices 15 to 27 is "(semi-group)"
# Note: If you want to keep parentheses, remove them from prose_punc.
# Assuming prose_punc = ",.!?;:" (Parentheses are NOT trimmed here)
tag_data = [HTMLTagWithIndices(tag, 15, 27)]
output = _consolidate_token_preds(main_text, tag_data)

test_eq(main_text[output[0].start: output[0].end], "(semi-group)")


# --- Test Case: Greedy Quote Marks ---
# Scenario: "Definition" - quotes should be trimmed if they surround the word.
main_text = 'We use the term "functor" to describe...'
tag = soup.new_tag('b', definition="")
# Indices 16 to 25 is '"functor"'
tag_data = [HTMLTagWithIndices(tag, 16, 25)]
# (Ensure \" is in your _trim_punctuation_from_indices list)
output = _consolidate_token_preds(main_text, tag_data)

test_eq(main_text[output[0].start: output[0].end], '\"functor\"')